# Unsloth Core — LoRA Training + GGUF Export

Train NPC LoRA adapter and export as GGUF for Unity/LLMUnity.

### What This Notebook Does

| Step | Description |
|------|-------------|
| **1. Setup** | Clone repo, install Unsloth, configure tokens |
| **2. Dataset Sync** | Find existing dataset or generate template fallback |
| **3. Train + Export** | Unsloth LoRA fine-tuning with inline GGUF export |
| **4. Verify** | Check exported GGUF files and adapter paths |
| **5. Download** | Download GGUF models to your local machine |

> **Requirements:** Google Colab with a GPU runtime (T4, L4, A100, or similar).
> If you are not running in Colab, open this notebook at
> [colab.research.google.com](https://colab.research.google.com/).


In [ ]:
# @title ⚙️ Configuration
# @markdown Configure the NPC and training parameters before running cells below.

# --- NPC Key ---
NPC_KEY = "history_guide"  # @param ["history_guide", "chef_assistant", "astronomy_guide", "fitness_coach"]
# @markdown The NPC key in snake_case.

# --- Dataset Technique ---
TECHNIQUE = "ollama"  # @param ["template", "ollama", "openai", "anthropic"]
# @markdown Technique used to generate the dataset (must already exist).
# @markdown - **template**: Fast deterministic (no LLM needed)
# @markdown - **ollama**: Local LLM-generated via Ollama
# @markdown - **openai**: OpenAI API-generated
# @markdown - **anthropic**: Anthropic API-generated

# --- Training Preset ---
PRESET = "fast-3b"  # @param ["smoke", "fast-3b", "safe-any", "premium-3b", "remote-3b-quality"]
# @markdown - **smoke**: Debug / quick test (~20 steps)
# @markdown - **fast-3b**: Standard NPC training (tuned for RTX 3060 6GB)
# @markdown - **safe-any**: Low VRAM fallback
# @markdown - **premium-3b**: Higher quality, more steps
# @markdown - **remote-3b-quality**: Best quality for remote Colab runtimes

# --- Hugging Face Token ---
HUGGINGFACE_TOKEN = ""  # @param {type:"string"}
# @markdown Read token for gated models (e.g. Llama 3.2). Get yours at
# @markdown https://huggingface.co/settings/tokens

# --- W&B API Key ---
WANDB_API_KEY = ""  # @param {type:"string"}
# @markdown Optional. Weights & Biases tracking key.
# @markdown Leave empty for offline mode (sync later).

print("Configuration loaded:")
print(f"  NPC_KEY       = {NPC_KEY}")
print(f"  TECHNIQUE     = {TECHNIQUE}")
print(f"  PRESET        = {PRESET}")
print(f"  HF token set  = {bool(HUGGINGFACE_TOKEN)}")
print(f"  WANDB key set = {bool(WANDB_API_KEY)}")


In [ ]:
# @title 📦 Setup: Clone Repository & Install Unsloth
# @markdown Mounts Google Drive, clones the repo, installs Unsloth.

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/athargamedev/Unsloth_Core.git"
DRIVE_REPO_DIR = "/content/drive/MyDrive/Unsloth_Core"
FALLBACK_REPO_DIR = "/content/Unsloth_Core"

# --- Detect Runtime ---
is_colab = False
try:
    import google.colab  # type: ignore
    is_colab = True
except ImportError:
    pass

if is_colab:
    print("Running in Google Colab.")
    repo_dir = DRIVE_REPO_DIR
    use_persistent = True
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        if os.path.exists("/content/drive/MyDrive"):
            print("Google Drive mounted. Using persistent storage:", repo_dir)
        else:
            raise OSError("Google Drive mounted but MyDrive is not accessible.")
    except Exception as e:
        repo_dir = FALLBACK_REPO_DIR
        use_persistent = False
        print(f"Drive mount failed: {e}")
        print("Using ephemeral storage:", repo_dir)

    # Clone or pull the repository
    try:
        if not os.path.exists(repo_dir):
            os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
            subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
            print("Cloned repository.")
        else:
            git_dir = os.path.join(repo_dir, ".git")
            if os.path.exists(git_dir) and os.path.isdir(git_dir):
                orig = os.getcwd()
                os.chdir(repo_dir)
                subprocess.run(["git", "pull"], check=False)
                os.chdir(orig)
                print("Pulled latest changes.")
            else:
                import shutil
                shutil.rmtree(repo_dir)
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
                print("Re-cloned repository (existing dir was not a git repo).")
    except OSError as e:
        print(f"Filesystem error during clone: {e}")
        if use_persistent:
            print("Falling back to ephemeral storage...")
            repo_dir = FALLBACK_REPO_DIR
            if not os.path.exists(repo_dir):
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)

    os.chdir(repo_dir)
    print("Changed to repo root:", os.getcwd())

    # --- VRAM Detection ---
    import torch
    vram_gb = 0.0
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
        gpu_name = torch.cuda.get_device_name(0)
        print(f"GPU: {gpu_name}  |  VRAM: {vram_gb:.2f} GB")
    else:
        print("WARNING: No GPU detected. Training will fail without CUDA.")

    # --- Install Unsloth ---
    print("\nInstalling Unsloth and dependencies...")
    subprocess.run(
        ["pip", "install", "--no-deps", "-q",
         "trl<0.9.0", "peft", "accelerate", "bitsandbytes", "xformers"],
        check=True,
    )
    subprocess.run(
        ["pip", "install", "-q",
         "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"],
        check=True,
    )
    subprocess.run(
        ["pip", "install", "-q", "gguf", "sentencepiece"],
        check=True,
    )
    print("All dependencies installed.")

    # --- Hugging Face Token ---
    if HUGGINGFACE_TOKEN:
        os.environ["HF_TOKEN"] = HUGGINGFACE_TOKEN
        try:
            from huggingface_hub import login  # type: ignore
            login(token=HUGGINGFACE_TOKEN, add_to_git_credential=True)
            print("Authenticated with Hugging Face Hub.")
        except ImportError:
            print("HF_TOKEN set as environment variable (huggingface_hub not installed).")
        except Exception as e:
            print(f"HF login failed: {e}")
    else:
        print("No HF token set. Gated models may return 401 Unauthorized.")

    # --- Weights & Biases ---
    if WANDB_API_KEY:
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY
        print("W&B API key configured.")
    else:
        os.environ.setdefault("WANDB_MODE", "offline")
        print("W&B API key not set. Using WANDB_MODE=offline.")
        print("  Sync later with: wandb sync wandb/offline-*")

else:
    print("=" * 60)
    print("NOT RUNNING IN GOOGLE COLAB.")
    print("Open this notebook at https://colab.research.google.com/")
    print("with a GPU runtime for the full pipeline.")
    print("=" * 60)
    print("Continuing in local environment (limited support).")
    curr = Path(os.getcwd()).resolve()
    repo_dir = None
    for parent in [curr] + list(curr.parents):
        if (parent / "ucore").exists() and (parent / "scripts").exists():
            repo_dir = parent
            break
    if repo_dir:
        print("Found local repo root:", repo_dir)
        os.chdir(str(repo_dir))
    else:
        print("Warning: Could not find local repo root.")

print("\nWorking directory:", os.getcwd())


In [ ]:
# @title 📂 Dataset Sync
# @markdown Find existing dataset or generate template dataset as fallback.

import os
import sys
import subprocess
from pathlib import Path

python_bin = sys.executable
dataset_dir = Path(f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}")
clean_jsonl = dataset_dir / "train_clean.jsonl"
raw_jsonl = dataset_dir / "train.jsonl"

print("=" * 60)
print("DATASET SYNC")
print(f"  NPC:       {NPC_KEY}")
print(f"  Technique: {TECHNIQUE}")
print(f"  Dataset:   {dataset_dir}")
print("=" * 60)

# --- Look for existing dataset ---
resolved_clean_path = clean_jsonl
resolved_raw_path = raw_jsonl

if clean_jsonl.exists():
    print(f"Found sanitized dataset: {clean_jsonl}")
elif raw_jsonl.exists():
    print(f"Found raw dataset: {raw_jsonl}")
    resolved_clean_path = raw_jsonl
    print("  Will use raw dataset (no sanitized version available).")
else:
    print(f"No dataset found for technique '{TECHNIQUE}'.")
    print("  Trying template technique as fallback...")
    template_raw = Path(f"subjects/datasets/{NPC_KEY}/template/train.jsonl")
    if template_raw.exists():
        print(f"Found template dataset: {template_raw}")
        resolved_clean_path = template_raw
    else:
        print("  No template dataset either. Generating template dataset now...")
        spec_path = f"subjects/{NPC_KEY}.json"
        cmd = [
            python_bin, "scripts/dataset/generate_dataset.py",
            spec_path, "--technique", "template",
        ]
        print("Running:", " ".join(cmd))
        try:
            subprocess.run(cmd, check=True)
            print("Template dataset generated.")
            resolved_clean_path = template_raw
        except subprocess.CalledProcessError as e:
            print(f"ERROR: Template generation failed: {e}")
            raise SystemExit(1)

print(f"\nDataset ready: {resolved_clean_path}")

# --- Validate as JSONL ---
print("\nValidating dataset format...")
try:
    subprocess.run(
        [python_bin, "-m", "json.tool", str(resolved_clean_path)],
        check=True, capture_output=True, timeout=15,
    )
except subprocess.CalledProcessError:
    print("WARNING: File is not valid JSON (expected for JSONL -- using head)")
    subprocess.run(["head", "-1", str(resolved_clean_path)])

line_count = 0
with open(resolved_clean_path, "r") as f:
    for _ in f:
        line_count += 1
size_kb = resolved_clean_path.stat().st_size / 1024
print(f"Dataset: {line_count} examples, {size_kb:.1f} KB")


In [ ]:
# @title 🏋️ Training + GGUF Export
# @markdown Run Unsloth LoRA fine-tuning and export as GGUF for Unity/LLMUnity.
# @markdown This matches the `ucore train ... --export-gguf` workflow.

import os
import sys
import subprocess
import json
import torch
import urllib.request
from pathlib import Path

python_bin = sys.executable
spec_path = f"subjects/{NPC_KEY}.json"
dataset_dir = Path(f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}")
clean_jsonl = dataset_dir / "train_clean.jsonl"
raw_jsonl = dataset_dir / "train.jsonl"

print("=" * 60)
print("STAGE: TRAINING + GGUF EXPORT")
print("=" * 60)

# --- Guard: spec must exist ---
if not Path(spec_path).exists():
    print(f"ERROR: Spec not found: {spec_path}")
    print("Run the Setup cell first.")
    raise SystemExit(1)

# --- Resolve dataset path ---
training_jsonl = clean_jsonl if clean_jsonl.exists() else raw_jsonl
if not training_jsonl.exists():
    print(f"ERROR: No dataset found in {dataset_dir}")
    print("Run the Dataset Sync cell first.")
    raise SystemExit(1)
print(f"Dataset: {training_jsonl}")

# --- VRAM Detection + Preset Downgrade ---
effective_preset = PRESET
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
    print(f"\nGPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")

    if vram_gb < 8 and effective_preset not in ("safe-any", "smoke"):
        print(f"  WARNING: Low VRAM ({vram_gb:.1f} GB).")
        if effective_preset == "fast-3b":
            print("  Auto-downgrading preset from 'fast-3b' to 'safe-any'.")
            effective_preset = "safe-any"
        elif effective_preset == "premium-3b":
            print("  Auto-downgrading preset from 'premium-3b' to 'fast-3b'.")
            effective_preset = "fast-3b"
elif vram_gb > 0:
    print(f"\nVRAM: {vram_gb:.2f} GB")
else:
    print("\nWARNING: No CUDA GPU detected. Training will fail.")

print(f"Effective preset: {effective_preset}")

# --- Unload Ollama models (free VRAM) ---
print("\nAttempting to unload Ollama models from GPU memory...")
try:
    payload = json.dumps({
        "model": "llama3.1:latest",
        "keep_alive": 0,
    }).encode("utf-8")
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=3):
        print("  Ollama models unloaded.")
except Exception:
    print("  Ollama not running or not reachable (continuing).")

# --- Build training command ---
cmd = [
    python_bin, "scripts/training/train.py",
    spec_path,
    "--from-spec",
    "--technique", TECHNIQUE,
    "--preset", effective_preset,
    "--export-gguf",
]

# Add W&B flag if API key is set
if os.environ.get("WANDB_API_KEY"):
    cmd.append("--wandb")
    print("\nW&B tracking enabled.")

print("\n" + "=" * 60)
print("TRAINING STARTED (may take 30-60 minutes)")
print("Running:", " ".join(cmd))
print("=" * 60)

# --- Execute training ---
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("\n" + "=" * 60)
    print("TRAINING FAILED")
    print("=" * 60)
    stdout_lines = result.stdout.strip().split("\n")
    print("\nLast 100 lines of stdout:")
    print("\n".join(stdout_lines[-100:]))
    stderr_lines = result.stderr.strip().split("\n")
    print("\nSTDERR (last 50 lines):")
    print("\n".join(stderr_lines[-50:]))
    raise SystemExit(1)

print(result.stdout)
print("\n" + "=" * 60)
print("TRAINING + GGUF EXPORT COMPLETE")
print("=" * 60)
print("\nProceed to the 'Export Verification' cell to inspect results.")


In [ ]:
# @title ✅ Export Verification
# @markdown Find and list exported GGUF files and adapter paths.

import os
from pathlib import Path

print("=" * 60)
print("EXPORT VERIFICATION")
print("=" * 60)

# --- Check GGUF exports ---
exports_dir = Path(f"exports/{NPC_KEY}")
if exports_dir.exists():
    gguf_files = sorted(exports_dir.glob("*.gguf"))
    if gguf_files:
        print(f"\nExported GGUF files in {exports_dir}:")
        for f in gguf_files:
            size_mb = f.stat().st_size / 1024**2
            print(f"  ✓ {f.name}  ({size_mb:.1f} MB)")
    else:
        print(f"\nNo .gguf files found in {exports_dir}.")
else:
    print(f"\nExport directory not found: {exports_dir}")
    print("Run the Training cell first.")

# --- Check adapter runs ---
runs_dir = Path(f"outputs/{NPC_KEY}/runs")
if runs_dir.exists():
    run_dirs = sorted(runs_dir.iterdir()) if runs_dir.is_dir() else []
    if run_dirs:
        latest_run = run_dirs[-1]
        print(f"\nLatest adapter run: {latest_run}")
        adapter_path = latest_run / "checkpoint-*/adapter_model.safetensors"
        adapter_files = sorted(latest_run.glob("**/adapter_model.safetensors"))
        if adapter_files:
            for af in adapter_files:
                size_mb = af.stat().st_size / 1024**2
                print(f"  ✓ {af.parent.name}/adapter_model.safetensors ({size_mb:.1f} MB)")
        else:
            print(f"  No adapter weights found in {latest_run.name}")
    else:
        print(f"\nNo run directories found in {runs_dir}")
else:
    print(f"\nRuns directory not found: {runs_dir}")

print("\nRun the Download cell to retrieve GGUF files.")


In [ ]:
# @title ⬇️ Download GGUF Files
# @markdown Download exported GGUF models from Colab to your local machine.

import os
import glob
from pathlib import Path

print("=" * 60)
print("DOWNLOAD ARTIFACTS")
print("=" * 60)

# --- Check runtime ---
is_colab = False
try:
    from google.colab import files  # type: ignore
    is_colab = True
except ImportError:
    pass

if not is_colab:
    print("Not running in Colab -- download via google.colab.files unavailable.")
    print("\nArtifacts can be found at:")
    print(f"  exports/{NPC_KEY}/")
    print(f"  outputs/{NPC_KEY}/runs/")
    raise SystemExit(0)

# --- Download GGUF files ---
gguf_dir = f"exports/{NPC_KEY}"
gguf_files = sorted(glob.glob(os.path.join(gguf_dir, "*.gguf")))

if not gguf_files:
    print(f"No GGUF files found in {gguf_dir}.")
    print("Run Training cell first.")
    raise SystemExit(0)

print(f"Found {len(gguf_files)} GGUF file(s):")
for f in gguf_files:
    size_mb = os.path.getsize(f) / 1024**2
    print(f"  {os.path.basename(f)}  ({size_mb:.1f} MB)")

print("\nDownloading GGUF files...")
for f in gguf_files:
    print(f"  -> {os.path.basename(f)}")
    files.download(f)

print("\nAll downloads complete.")
print("Copy the GGUFs to your Unity project's StreamingAssets directory.")
